# Calidad del vino tinto

**Sección 3.8 y 3.9 del libro**

El mismo programa del iris, ahora con 11 propiedades químicas para predecir la calidad del vino (0 a 10). Reúne los ejemplos `K3_redwine-1`, `-2` y `-3` del libro.

> **Basado en:** Karatas, M. (2024). *Developing AI Applications: An Introduction*, cap. 3 «Artificial Neural Networks». Rheinwerk Publishing. Código original del autor, adaptado a Google Colab y comentado en español con fines educativos.


In [ ]:
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split

## Cargar los datos

Antes de ejecutar esta celda, sube **winequality-red.csv** al panel de archivos de Colab (ícono de carpeta → botón de subir). Si no lo tienes a mano, usa la línea comentada: lee el archivo directamente desde la página de la presentación.

In [ ]:
path = "winequality-red.csv"
# path = "https://sinergia-iesa.github.io/introduccion-a-la-creacion-de-redes-neuronales/data/winequality-red.csv"   # alternativa sin subir el archivo

data = pd.read_csv(path, delimiter=';')
print(data.head())
print("Columnas con huecos:", data.columns[data.isnull().any()])

## Versión 1: 11 clases (calidad 0 a 10)

Cambian tres cosas respecto al iris: el archivo, la columna objetivo y el tamaño de la red.

In [ ]:
col = data['quality']
X = data.drop(['quality'], axis=1)

train_data, test_data, train_col, test_col = train_test_split(X, col, test_size=0.2, random_state=42)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(11,)),                       # 12 columnas − 1 objetivo
    tf.keras.layers.Dense(32, activation='sigmoid'),
    tf.keras.layers.Dense(64, activation='sigmoid'),
    tf.keras.layers.Dense(11, activation='softmax')    # calidades 0…10
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=70, verbose=0)
print('Test accuracy (11 clases):', model.evaluate(test_data, test_col, verbose=0)[1])

Ronda el 55–60 %. Personas probaron y calificaron los vinos: no es una ciencia exacta.

## Versión 2: solo 3 clases

¿Hace falta distinguir 11 niveles? Agrupamos en **malo (0)**, **bueno (1)** y **muy bueno (2)**. El libro lo hace en una hoja de cálculo; aquí lo hacemos con pandas.

In [ ]:
col3 = pd.cut(data['quality'], bins=[-1, 5, 6, 10], labels=[0, 1, 2]).astype(int)
print(col3.value_counts())

train_data, test_data, train_col, test_col = train_test_split(X, col3, test_size=0.2, random_state=42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(11,)),
    tf.keras.layers.Dense(32, activation='sigmoid'),
    tf.keras.layers.Dense(60, activation='sigmoid'),
    tf.keras.layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=70, verbose=0)
print('Test accuracy (3 clases):', model.evaluate(test_data, test_col, verbose=0)[1])

## Versión 3: jugar con los hiperparámetros

Activación `relu`, una capa oculta más y 50 épocas. `random_state=42` hace que la mezcla de datos sea siempre igual, para comparar de forma justa.

In [ ]:
train_data, test_data, train_col, test_col = train_test_split(X, col, test_size=0.2, random_state=42)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(11,)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(120, activation='relu'),
    tf.keras.layers.Dense(60, activation='relu'),
    tf.keras.layers.Dense(11, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(train_data, train_col, epochs=50, verbose=0)
print('Test accuracy (relu, 3 capas):', model.evaluate(test_data, test_col, verbose=0)[1])